In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/dylanmalis-collab/Exoplanet_Life/refs/heads/elt_v2/Exoplanet_DBT/data_exports/raw_exoplanets_enriched.csv"
df = pd.read_csv(url)

C:\Users\berna\AppData\Local\Temp\ipykernel_18184\287856858.py:3: DtypeWarning: Columns (0: hd_name, 1: pl_orbtperstr, 2: pl_orbtper_reflink, 3: pl_orbtper_systemref, 4: pl_trueobliqstr, 5: pl_trueobliq_reflink, 6: st_log_rhkstr, 7: st_log_rhk_reflink, 8: sy_icmagstr, 9: pl_cmassjstr, 10: pl_cmassj_reflink, 11: pl_cmassestr, 12: pl_cmasse_reflink) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)


In [2]:
#première selection grossière
df = df[["pl_name", "hostname", "pl_orbsmax_calc","pl_rade", "pl_bmasse_calc", "pl_dens_calc", "pl_orbeccen", "pl_insol_calc", "pl_eqt_calc","st_spectype_group_en", "st_teff", "st_rad", "st_mass", "st_met","st_lum", "st_lum_lin_calc", "st_dens", "Gravity_G_earth_calc", "masse_10_24kg_calc","pl_type", "sim_earth", "eau_possible","pt_habitable"]]

In [3]:
#transformation (Nan et création des colonne missing)
df["pl_orbeccen"] = df["pl_orbeccen"].fillna(df["pl_orbeccen"].median())

df["st_teff_missing"] = df["st_teff"].isna().astype(int)

# 1. imputation par groupe
df["st_teff"] = df.groupby("st_spectype_group_en")["st_teff"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_teff"] = df["st_teff"].fillna(df["st_teff"].median())

df["st_mass_missing"] = df["st_mass"].isna().astype(int)

# 1. imputation par groupe
df["st_mass"] = df.groupby("st_spectype_group_en")["st_mass"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_mass"] = df["st_mass"].fillna(df["st_mass"].median())

df["st_rad_missing"] = df["st_rad"].isna().astype(int)

# 1. imputation par groupe
df["st_rad"] = df.groupby("st_spectype_group_en")["st_rad"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_rad"] = df["st_rad"].fillna(df["st_rad"].median())

df["st_met_missing"] = df["st_met"].isna().astype(int)

# 1. imputation par groupe
df["st_met"] = df.groupby("st_spectype_group_en")["st_met"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_met"] = df["st_met"].fillna(df["st_met"].median())

df["st_dens_missing"] = df["st_dens"].isna().astype(int)

# 1. imputation par groupe
df["st_dens"] = df.groupby("st_spectype_group_en")["st_dens"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_dens"] = df["st_dens"].fillna(df["st_dens"].median())

df["st_lum_calc"] = df["st_lum"].fillna(df.groupby("st_spectype_group_en")["st_lum"].transform("median"))

In [40]:
# juste pl_insol
df4 = df[["pl_rade", "pl_bmasse_calc", "pl_dens_calc", "sim_earth","pl_insol_calc"]]

In [41]:
from sklearn.model_selection import train_test_split

X = df4
y = df["pt_habitable"]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y, random_state=42)

In [42]:
import sys
print(sys.executable)

j:\Projet 3\.venv\Scripts\python.exe


In [43]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier
#from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
cv = RepeatedStratifiedKFold( n_splits=5, n_repeats=3, random_state=42 )

In [44]:
model_xgb = XGBClassifier(random_state=42,tree_method="hist", colsample_bytree = 0.5, gamma = 0, learning_rate = 0.05, max_delta_step = 1, max_depth = 5, min_child_weight = 1, n_estimators = 300, reg_alpha = 0.1, reg_lambda = 5, scale_pos_weight = 50, subsample= 1.0)
model_xgb.fit(X_train, y_train)


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.5
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [45]:
from sklearn.metrics import f1_score, recall_score, average_precision_score, precision_score

y_proba = model_xgb.predict_proba(X_test)[:, 1]
y_pred = model_xgb.predict(X_test)

f1 = f1_score(y_test, y_pred)
print("F1 score (threshold=0.5):", f1)

recall = recall_score(y_test, y_pred)
print("Recall (threshold=0.5):", recall)

precision = precision_score(y_test, y_pred)
print("Precision (threshold=0.5):", precision)

ap = average_precision_score(y_test, y_proba)
print("Average Precision:", ap)


F1 score (threshold=0.5): 0.6666666666666666
Recall (threshold=0.5): 0.5
Precision (threshold=0.5): 1.0
Average Precision: 1.0


In [46]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1230
           1       1.00      0.50      0.67         2

    accuracy                           1.00      1232
   macro avg       1.00      0.75      0.83      1232
weighted avg       1.00      1.00      1.00      1232



In [47]:
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.base import clone

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

oof_proba = np.zeros(len(X_train))

for train_idx, val_idx in cv.split(X_train, y_train):
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = clone(model_xgb)  # important !
    model.fit(X_tr, y_tr)

    oof_proba[val_idx] = model.predict_proba(X_val)[:, 1]

In [48]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_train, oof_proba)

precision = precision[:-1]
recall = recall[:-1]

target_precision = 0.8  # ou 0.9 selon ton objectif

valid_idx = np.where(precision >= target_precision)[0]

best_idx = valid_idx[np.argmax(recall[valid_idx])]
best_threshold = thresholds[best_idx]

print("Best threshold (OOF):", best_threshold)
print("Precision:", precision[best_idx])
print("Recall:", recall[best_idx])

Best threshold (OOF): 0.7201443910598755
Precision: 0.8571428571428571
Recall: 0.6666666666666666


In [49]:
target_recall = 0.9

valid_idx = np.where(recall >= target_recall)[0]

best_idx = valid_idx[np.argmax(precision[valid_idx])]
best_threshold = thresholds[best_idx]

best_precision = precision[best_idx]
best_recall = recall[best_idx]

print("Threshold (recall ≥ 0.9):", best_threshold)
print("Precision (habitable):", best_precision)
print("Recall (habitable):", best_recall)

Threshold (recall ≥ 0.9): 0.03746724873781204
Precision (habitable): 0.34615384615384615
Recall (habitable): 1.0


In [39]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_train, oof_proba)

precision = precision[:-1]
recall = recall[:-1]

target_precision = 0.6

valid_idx = np.where(
    (recall == 1.0) & (precision >= target_precision)
)[0]

if len(valid_idx) > 0:
    best_idx = valid_idx[np.argmax(precision[valid_idx])]
    best_threshold = thresholds[best_idx]

    print("Threshold:", best_threshold)
    print("Precision:", precision[best_idx])
    print("Recall:", recall[best_idx])
else:
    print("Impossible: recall=1 et precision>=0.7")

Impossible: recall=1 et precision>=0.7


In [50]:
import pandas as pd

data = [

    # 🌍 Terre (positif)
    [1.00, 1.00, 5.51,
     1.0, 1.0],

    # 🪨 Mercure
    [ 0.383, 0.055, 5.43,
     0.15, 9.1],

    # ♀ Venus
    [ 0.949, 0.815, 5.24,
     0.85, 1.9],

    # 🔴 Mars
    [0.532, 0.107, 3.93,
     0.75, 0.43],

    # 🪐 Jupiter
    [11.21, 317.8, 1.33,
     0.10, 0.036],

    # 🪐 Saturne
    [9.45, 95.2, 0.69,
     0.08, 0.011],

    # 🧊 Uranus
    [ 4.01, 14.5, 1.27,
     0.03, 0.0037],

    # 🧊 Neptune
    [ 3.88, 17.1, 1.64,
     0.02, 0.0015],

]

columns = [
    "pl_rade","pl_bmasse_calc","pl_dens_calc",
    "sim_earth","pl_insol_calc"
]

X_solar = pd.DataFrame(data, columns=columns)

y_solar = [1,0,0,0,0,0,0,0]

In [51]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
import numpy as np

threshold = 0.5

# Probabilités classe 1
y_proba_solar = model_xgb.predict_proba(X_solar)[:, 1]

# Prédictions avec ton threshold
y_pred_solar = (y_proba_solar >= threshold).astype(int)

# Metrics classiques (classe 1 = habitable)
precision = precision_score(y_solar, y_pred_solar)
recall = recall_score(y_solar, y_pred_solar)
f1 = f1_score(y_solar, y_pred_solar)

# Average Precision (utilise les proba, pas le threshold)
ap = average_precision_score(y_solar, y_proba_solar)

print("Threshold:", threshold)
print("Precision (class 1):", precision)
print("Recall (class 1):", recall)
print("F1 score:", f1)
print("Average Precision:", ap)

Threshold: 0.5
Precision (class 1): 1.0
Recall (class 1): 1.0
F1 score: 1.0
Average Precision: 1.0


In [62]:
import numpy as np
import pandas as pd

np.random.seed(42)

def generate_controlled_planets(n_total=5000, pos_ratio=0.2):

    n_pos = int(n_total * pos_ratio)
    n_neg = n_total - n_pos

    data = []

    # =====================================================
    # 🌍 POSITIVES (habitables selon TES règles)
    # =====================================================
    for _ in range(n_pos):

        pl_rade = np.random.uniform(0.8, 1.7)
        pl_bmasse_calc = np.random.uniform(0.6, 4.0)
        min_dens = 2.2 / ((1 / pl_rade) ** 3)

        # on force au-dessus du minimum
        pl_dens_calc = np.random.uniform(min_dens, min_dens + 1.5)
        pl_orbsmax_calc = np.random.uniform(0.8, 1.5)
        pl_orbeccen = np.random.uniform(0, 0.2)

        st_spectype_group_en = np.random.choice(["G", "K", "F"])  # favorables

        st_teff = np.random.choice([5000, 5778, 6000])
        st_rad = np.random.uniform(0.8, 1.2)
        st_mass = np.random.uniform(0.8, 1.2)
        st_met = np.random.uniform(-0.2, 0.3)

        st_lum_calc = np.random.uniform(-0.3, 0.2)  # log solar
        st_dens = np.random.uniform(1.0, 2.0)

        # derived
        st_lum_lin = 10 ** st_lum_calc
        pl_insol_calc = np.random.uniform(0.5, 1.5)

        Gravity_G_earth_calc = pl_bmasse_calc / (pl_rade ** 2)

        eau_possible = 1

        pl_type = "Rocheuse"

        # SCORE ROCH implicite
        Score_roch = 3.0  # forcé > 2.2

        pl_eqt_calc = 255 * (pl_insol_calc ** 0.25)

        pt_habitable = 1

        data.append([
            pl_orbsmax_calc, pl_rade, pl_bmasse_calc, pl_dens_calc,
            pl_orbeccen,
            st_teff, st_rad, st_mass, st_met,
            st_lum_calc, st_dens,
            st_spectype_group_en,
            pl_insol_calc,
            Gravity_G_earth_calc,
            eau_possible,
            pl_type,
            pl_eqt_calc,
            Score_roch,
            pt_habitable
        ])

    # =====================================================
    # ☠️ NEGATIVES (non habitables variés)
    # =====================================================
    for _ in range(n_neg):

        pl_rade = np.random.uniform(0.2, 0.3)
        pl_bmasse_calc = np.random.uniform(0.05, 0.5)
        pl_dens_calc = np.random.uniform(0.5, 3)
        pl_orbsmax_calc = np.random.uniform(0.05, 5.0)
        pl_orbeccen = np.random.uniform(0, 0.8)

        st_spectype_group_en = np.random.choice(["M", "G", "K", "F"])

        st_teff = np.random.choice([3000, 4000, 5000, 6000, 6500])
        st_rad = np.random.uniform(0.2, 2.0)
        st_mass = np.random.uniform(0.2, 2.0)
        st_met = np.random.uniform(-1, 0.5)

        st_lum_calc = np.random.uniform(-3, 1)
        st_dens = np.random.uniform(0.5, 5.0)

        st_lum_lin = 10 ** st_lum_calc
        pl_insol_calc = st_lum_lin / (pl_orbsmax_calc ** 2)

        Gravity_G_earth_calc = pl_bmasse_calc / (pl_rade ** 2)

        eau_possible = int(0)  # bruit volontaire

        pl_type = np.random.choice(["Gazeuse/Mixte", "Rocheuse"])

        Score_roch = np.random.uniform(0, 2.5)  # souvent < seuil

        pl_eqt_calc = 255 * (pl_insol_calc ** 0.25)

        pt_habitable = 0

        data.append([
            pl_orbsmax_calc, pl_rade, pl_bmasse_calc, pl_dens_calc,
            pl_orbeccen,
            st_teff, st_rad, st_mass, st_met,
            st_lum_calc, st_dens,
            st_spectype_group_en,
            pl_insol_calc,
            Gravity_G_earth_calc,
            eau_possible,
            pl_type,
            pl_eqt_calc,
            Score_roch,
            pt_habitable
        ])

    # =====================================================
    # DATAFRAME
    # =====================================================
    cols = [
        "pl_orbsmax_calc","pl_rade","pl_bmasse_calc","pl_dens_calc",
        "pl_orbeccen",
        "st_teff","st_rad","st_mass","st_met",
        "st_lum_calc","st_dens",
        "st_spectype_group_en",
        "pl_insol_calc",
        "Gravity_G_earth_calc",
        "eau_possible",
        "pl_type",
        "pl_eqt_calc",
        "Score_roch",
        "pt_habitable"
    ]

    return pd.DataFrame(data, columns=cols)

df_random2 = generate_controlled_planets(n_total=5000, pos_ratio=0.2)

df_random2["sim_earth"] = (
    (1 - np.abs((df["pl_rade"] - 1) / (df["pl_rade"] + 1))) ** 0.57 *
    (1 - np.abs((df["pl_dens_calc"] - 5.51) / (df["pl_dens_calc"] + 5.51))) ** 1.07 *
    (1 - np.abs((df["Gravity_G_earth_calc"] - 1) / (df["Gravity_G_earth_calc"] + 1))) ** 0.70 *
    (1 - np.abs((df["pl_eqt_calc"] - 288) / (df["pl_eqt_calc"] + 288))) ** 5.58
) ** 0.25

X_random2 = df_random2[["pl_rade", "pl_bmasse_calc", "pl_dens_calc", "sim_earth","pl_insol_calc"]]
y_random2 = df_random2["pt_habitable"]

In [59]:
df_random2["pt_habitable"].value_counts()

pt_habitable
0    4000
1    1000
Name: count, dtype: int64

In [60]:
df_random2["sim_earth"].sort_values(ascending=False).head()

4962    0.965170
4379    0.951177
3940    0.948194
2607    0.942654
1957    0.931013
Name: sim_earth, dtype: float64

In [63]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

# ton threshold perso
threshold = 0.01

# probabilités classe 1
y_proba_random2 = model_xgb.predict_proba(X_random2)[:, 1]

# prédictions avec threshold custom
y_pred_random2 = (y_proba_random2 >= threshold).astype(int)

# =========================
# METRICS
# =========================

precision = precision_score(y_random2, y_pred_random2)
recall = recall_score(y_random2, y_pred_random2)
f1 = f1_score(y_random2, y_pred_random2)

# average precision (indépendant du threshold)
ap = average_precision_score(y_random2, y_proba_random2)

print("Threshold:", threshold)
print("Precision (class 1):", precision)
print("Recall (class 1):", recall)
print("F1 score:", f1)
print("Average Precision:", ap)

Threshold: 0.01
Precision (class 1): 0.8567901234567902
Recall (class 1): 0.347
F1 score: 0.49395017793594304
Average Precision: 0.522577544932483


In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

y_proba_train1 = model_xgb.predict_proba(X_train)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, y_proba_train1)

# alignement
precision = precision[:-1]
recall = recall[:-1]

# condition : precision >= 0.8
target_precision = 0.85

valid_idx = np.where(precision >= target_precision)[0]

if len(valid_idx) > 0:
    # on prend celui avec le meilleur recall
    best_idx = valid_idx[np.argmax(recall[valid_idx])]
    
    best_threshold = thresholds[best_idx]
    
    print("Best threshold:", best_threshold)
    print("Precision:", precision[best_idx])
    print("Recall:", recall[best_idx])
else:
    print("Impossible d’atteindre precision >= 0.85")

Best threshold: 0.21271472
Precision: 0.9
Recall: 1.0


In [45]:
y_proba_test = model_xgb.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= 0.042273085564374924).astype(int)

from sklearn.metrics import precision_score, recall_score

print("Precision test:", precision_score(y_test, y_pred_test))
print("Recall test:", recall_score(y_test, y_pred_test))

Precision test: 0.2857142857142857
Recall test: 1.0


In [13]:
from sklearn.metrics import precision_recall_curve
import numpy as np

y_proba_train = model_xgb.predict_proba(X_train)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, y_proba_train)

# enlever dernier point (comme d’hab)
recall = recall[:-1]
precision = precision[:-1]

# index du recall max
best_idx = np.argmax(recall)

best_threshold = thresholds[best_idx]

print("Best threshold (max recall):", best_threshold)
print("Recall:", recall[best_idx])
print("Precision:", precision[best_idx])

Best threshold (max recall): 7.012817e-05
Recall: 1.0
Precision: 0.0018270401948842874


In [11]:
from sklearn.base import clone
import numpy as np
from sklearn.metrics import precision_recall_curve

cv = cv

oof_proba = np.zeros(len(X_train))

for train_idx, val_idx in cv.split(X_train, y_train):

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr = y_train.iloc[train_idx]

    model = clone(model_xgb)
    model.fit(X_tr, y_tr)

    oof_proba[val_idx] = model.predict_proba(X_val)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, oof_proba)

f1_scores = (2 * precision * recall) / (precision + recall + 1e-9)
f1_scores = f1_scores[:-1]

best_threshold = thresholds[np.argmax(f1_scores)]

print("Best threshold (CV):", best_threshold)

Best threshold (CV): 0.8826188445091248


In [12]:
y_pred_test = (y_proba >= best_threshold).astype(int)

print("F1 test:", f1_score(y_test, y_pred_test))

F1 test: 0.6666666666666666


In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

y_proba_train = model_xgb.predict_proba(X_train)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, y_proba_train)

f1_scores = (2 * precision * recall) / (precision + recall + 1e-9)
f1_scores = f1_scores[:-1]

best_idx = np.argmax(f1_scores)
best_threshold2 = thresholds[best_idx]

print("Best threshold (train):", best_threshold2)

In [ ]:
from sklearn.metrics import f1_score

y_proba_test = model_xgb.predict_proba(X_test)[:, 1]

y_pred_test = (y_proba_test >= best_threshold).astype(int)

print("F1 test:", f1_score(y_test, y_pred_test))

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model_xgb,
    X_train,
    y_train,
    cv=cv,
    scoring="average_precision",
    n_jobs=-1
)


In [ ]:
print("\n📈 CV stability (Average Precision)")
print("Mean:", cv_scores.mean())
print("Std :", cv_scores.std())

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

threshold = 0.1466639311252936  # ton seuil optimisé

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

for train_idx, val_idx in cv.split(X_train, y_train):

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = clone(model_xgb)
    model.fit(X_tr, y_tr)

    y_proba = model.predict_proba(X_val)[:, 1]
    y_pred3 = (y_proba >= threshold).astype(int)

    f1 = f1_score(y_val, y_pred3)
    f1_scores.append(f1)

print("F1 CV mean:", np.mean(f1_scores))
print("F1 CV std :", np.std(f1_scores))